# Fine-tune the intent router on Colab

Runtime -> Change runtime type -> **T4 GPU** before running anything.

The Mac path is not viable: at fp32 on MPS a training step took 74-166 s, which is
17-39 hours for this run. On a T4 with fp16 the same run is roughly 10-15 minutes.


## 1. Confirm the GPU

If this prints `cpu`, stop and switch the runtime type.


In [ ]:
import torch
print('cuda available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')


## 2. Get the code

Set `REPO` to your repository, then run.


In [ ]:
REPO = 'https://github.com/Mann-gupta1/agentic-support-triage.git'

import os, pathlib
if not pathlib.Path('agentic-support-triage').exists():
    !git clone -q $REPO
os.chdir('agentic-support-triage')
!ls


## 3. Install

Colab already ships torch with CUDA, so it is left out here on purpose -
reinstalling it pulls a CPU wheel and silently kills the GPU path.


In [ ]:
!pip install -q 'transformers>=4.44' 'peft>=0.12' 'accelerate>=0.33' \
    'datasets>=2.20' 'scikit-learn>=1.5' 'langgraph>=0.2' 'chromadb>=0.5' python-dotenv


## 4. Smoke run first

One epoch, so a broken pipeline costs 4 minutes instead of 15. Validation
accuracy should be far above chance (1/77 = 0.013) by the end of it.


In [ ]:
!LORA_EPOCHS=1 python -m src.finetune


## 5. Full run

Three epochs. This overwrites the one-epoch adapter.


In [ ]:
!python -m src.finetune


## 6. Evaluate both arms

This is the number that matters: the tuned router against the classical
baseline, on the same sampled test set.


In [ ]:
!python -m src.evaluate --arms tfidf,lora --limit 500


## 7. Read the report


In [ ]:
import glob
latest = sorted(glob.glob('reports/eval-*.md'))[-1]
print(latest)
print(open(latest).read())


## 8. Download the adapter and the report

The adapter is a few MB - it is only the LoRA weights, not the base model.
Commit `reports/` back to the repo so the README numbers have a source.


In [ ]:
!zip -qr router_artifacts.zip artifacts/lora-router reports
from google.colab import files
files.download('router_artifacts.zip')
